# Proyecto Integrador — Catastro Bogotá
## Descubrimiento de tipologías y predicción de variables catastrales mediante Machine Learning

**Asignatura:** Ciencia de Datos e Inteligencia Artificial · Maestría en Inteligencia Artificial · Universidad de La Salle

**Autor:** Ivan Alejandro Ceballos Galvis · **Docente:** Fabian Castro Riveros

---

Este notebook ejecuta de principio a fin el pipeline del proyecto, siguiendo la metodología **CRISP-DM** y las fases definidas en la guía técnica:

| Fase | Contenido |
|------|-----------|
| **Fase 1** | Planteamiento del problema, EDA y diagnóstico de calidad de datos |
| **Fase 2** | Limpieza, imputación, ingeniería de variables y modelo base (baseline) |
| **Fase 3** | Modelado avanzado: regresión, clasificación y clustering + evaluación |
| **Fase 4** | Resultados, interpretación y exportación para el dashboard |

> **Nota sobre el dataset:** el archivo completo `TPREDIO.csv` (~1.4 GB, ~3.8 M de predios) **no** se versiona en GitHub por su tamaño. Este notebook lo carga desde Google Drive, desde una subida manual, o desde la muestra `data/sample/TPREDIO_sample.csv` (6.000 filas) incluida en el repositorio.


## 0. Configuración inicial

Se importan todas las librerías y se fijan las constantes del proyecto. Google Colab ya incluye `pandas`, `numpy`, `matplotlib`, `seaborn` y `scikit-learn`; si usas una versión antigua, descomenta la primera línea para actualizar.


In [ ]:
# !pip install -q -U scikit-learn pandas numpy matplotlib seaborn

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---- scikit-learn ---------------------------------------------------------
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit

# Regresión
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

# Clustering
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering

# Métricas
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, silhouette_score, confusion_matrix, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

import sklearn
print(f"scikit-learn {sklearn.__version__}  |  pandas {pd.__version__}  |  numpy {np.__version__}")


In [ ]:
# =============================================================================
# Constantes del proyecto
# =============================================================================
RANDOM_STATE = 42
TEST_SIZE = 0.20

# Columnas de códigos catastrales -> se tratan como texto (identificadores)
COLS_CODIGO = ["PRECBARRIO", "PRECMANZ", "PRECPREDIO", "PRECCONS", "PRECRESTO"]

# Variables industriales y de acabados: nulos masivos -> imputar con 0 ("no aplica")
COLS_INDUSTRIALES = ["PRECINDUS", "PREACERCHA"]
COLS_ACABADOS = [
    "PREECONS", "PREAFACHAD", "PREACUBIER", "PREAPISOS", "PREACONS",
    "PREBTAMANO", "PREBENCHAP", "PREBMOBILI", "PREBCONS",
    "PRECTAMANO", "PRECENCHAP", "PRECMOBILI", "PRECCONSE",
]
COLS_ESTRUCTURA = ["PREEARMAZ", "PREEMUROS", "PREECUBIER"]

# Objetivos
TARGET_REGRESION = "PREACONST"        # área construida (m²)
TARGET_CLASIFICACION = "PRECDESTIN"   # destino económico -> binario
CLASE_RESIDENCIAL = 1                 # código 1 = uso habitacional

# Filtrado de atípicos para modelado
AREA_MIN, AREA_MAX = 0.0, 10_000.0
PUNTAJE_MIN = 0.0

# Para agilizar el modelado en CPU gratuita de Colab se usa una muestra.
# Cambia a None para usar TODAS las filas filtradas (más lento).
MAX_ROWS_FOR_MODELING = 200_000

# Variables predictoras base
FEATURES_NUM = ["PREATERRE", "PREPUNTAJE", "PREEARMAZ", "PREEMUROS", "PREECUBIER",
                "IND_CONST", "CALIDAD_EST"]
FEATURES_CLUSTER = ["PREATERRE", "PREACONST", "PREPUNTAJE", "PREEARMAZ", "PREEMUROS", "PREECUBIER"]

print("Configuración lista.")


## 1. Carga del dataset

Ejecuta **una sola** de las siguientes opciones para disponer del dataset:

- **Opción A (Drive):** monta Google Drive y lee `TPREDIO.csv` desde ahí.
- **Opción B (subir):** sube el archivo directamente a la sesión de Colab.
- **Opción C (muestra):** usa la muestra del repositorio (6.000 filas) para pruebas.

Después de elegir, la variable `DF` queda lista con los datos.


In [ ]:
# ---------------------------------------------------------------------------
# Opción A: montar Google Drive y leer el CSV completo
# ---------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

RUTAS_CANDIDATAS = [
    '/content/drive/MyDrive/proyecto_datos_urbanos/data/TPREDIO.csv',
    '/content/drive/MyDrive/TPREDIO.csv',
    'data/TPREDIO.csv',                    # si clonaste el repo y subiste el CSV completo
    'data/sample/TPREDIO_sample.csv',      # muestra de 6.000 filas (incluida en el repo)
]

RUTA = None
for r in RUTAS_CANDIDATAS:
    if os.path.exists(r):
        RUTA = r
        break

if RUTA is None:
    print("No se encontró ningún CSV. Sube el archivo con la Opción B.")
else:
    print("Usando:", RUTA)


In [ ]:
# ---------------------------------------------------------------------------
# Opción B (alternativa): subir TPREDIO.csv o TPREDIO_sample.csv manualmente
# Descomenta para usar esta opción en lugar de Drive.
# ---------------------------------------------------------------------------
# from google.colab import files
# uploaded = files.upload()          # elige el archivo en el diálogo
# RUTA = list(uploaded.keys())[0]
# print("Subido:", RUTA)


In [ ]:
# ---------------------------------------------------------------------------
# Carga efectiva del DataFrame
# ---------------------------------------------------------------------------
if RUTA is None:
    raise FileNotFoundError(
        "No hay dataset disponible. Monta Google Drive (Opción A) o sube el "
        "archivo (Opción B) antes de ejecutar esta celda."
    )

COLS_CODIGO = ["PRECBARRIO", "PRECMANZ", "PRECPREDIO", "PRECCONS", "PRECRESTO"]
# Códigos largos que deben leerse como texto (PRENUPRE tiene 30 dígitos)
COLS_CODIGO_LARGO = ["PRENUPRE", "PRECZHF"]

dtype = {c: str for c in COLS_CODIGO + COLS_CODIGO_LARGO}

DF = pd.read_csv(
    RUTA,
    sep=",",
    dtype=dtype,
    low_memory=False,
)
DF.columns = [c.strip() for c in DF.columns]

print(f"Filas: {len(DF):,}  |  Columnas: {DF.shape[1]}")
DF.head()


## FASE 1 — Análisis Exploratorio y Diagnóstico de Calidad

### 1.1 Estructura, tipos de dato y completitud


In [ ]:
print("=== Estructura ===")
print(f"Forma: {DF.shape}")
DF.info()

print("\n=== Valores nulos (absolutos y porcentaje) ===")
nulos = DF.isnull().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
if len(nulos):
    resumen = pd.DataFrame({
        "nulos": nulos,
        "pct": (nulos / len(DF)) * 100,
    })
    display(resumen)
else:
    print("Sin valores nulos.")

print("\n=== Duplicados ===")
print("Duplicados exactos:", DF.duplicated().sum())
print("Duplicados por CHIP (llave primaria):", DF.duplicated(subset=['PRECHIP']).sum())


In [ ]:
# Estadísticas descriptivas de las variables numéricas más relevantes
cols_num = ['PREATERRE', 'PREACONST', 'PREPUNTAJE', 'PREEARMAZ', 'PREEMUROS', 'PREECUBIER']
DF[cols_num].describe().T.round(3)


### 1.2 Análisis univariado (distribuciones y frecuencias)


In [ ]:
# Distribución de las áreas (con escala logarítmica por su alta asimetría)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, col, color in zip(axes, ['PREATERRE', 'PREACONST'], ['skyblue', 'salmon']):
    v = DF[col].clip(lower=0, upper=DF[col].quantile(0.99))
    ax.hist(v, bins=60, color=color, alpha=0.85)
    ax.set_title(f'Distribución de {col}')
    ax.set_xlabel('m²')
    ax.set_ylabel('Frecuencia')
plt.tight_layout()
plt.show()


In [ ]:
# Frecuencia de las principales variables categóricas
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['PRECDESTIN', 'PREUCALIF', 'PRECLASE']):
    order = DF[col].astype(str).value_counts().head(12).index
    sns.countplot(data=DF, x=col, order=order, ax=ax, palette='viridis')
    ax.set_title(f'Frecuencia de {col}')
    ax.tick_params(axis='x', rotation=60)
plt.tight_layout()
plt.show()


### 1.3 Análisis bivariado y multivariado (correlación y dispersión)


In [ ]:
# Matriz de correlación (Pearson) de las variables numéricas clave
corr = DF[['PREATERRE', 'PREACONST', 'PREPUNTAJE', 'PREEARMAZ', 'PREEMUROS', 'PREECUBIER']].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', center=0, square=True)
plt.title('Matriz de correlación (Pearson)')
plt.show()


In [ ]:
# Relación área de terreno vs área construida (muestra para aligerar la gráfica)
muestra_plot = DF[(DF['PREATERRE'] > 0) & (DF['PREATERRE'] < 5000) &
                  (DF['PREACONST'] > 0) & (DF['PREACONST'] < 5000)].sample(3000, random_state=RANDOM_STATE)
plt.figure(figsize=(7, 6))
sns.scatterplot(data=muestra_plot, x='PREATERRE', y='PREACONST', alpha=0.25, s=12)
plt.title('Área construida vs Área de terreno')
plt.xlabel('Área de terreno (m²)')
plt.ylabel('Área construida (m²)')
plt.show()


### 1.4 Diagnóstico de calidad y hallazgos

Del perfilado se desprenden los siguientes hallazgos (documentados en el informe técnico):

1. **Nulos "condicionales" masivos** en `PREACERCHA`/`PRECINDUS` (~74 %) y en acabados (~27 %): en catastro significan *"no aplica"* (el predio no tiene ese uso/calificación). Se imputan con **0**, no se eliminan.
2. **Códigos catastrales** (`PRECBARRIO`, `PRECMANZ`, …) leídos como número: deben convertirse a **texto** para no perder ceros a la izquierda.
3. **`PREPUNTAJE == -1`**: comodín para predios exentos/sin calificar; se excluye del modelado.
4. **`PREATERRE == 0`**: normal en Propiedad Horizontal (el terreno está a nombre de la matriz, no del apartamento).


## FASE 2 — Limpieza, Ingeniería de Variables y Modelo Base

### 2.1 Limpieza e imputación (con justificación técnica)


In [ ]:
def limpiar(df):
    """Aplica la limpieza documentada del dataset."""
    out = df.copy()

    # 1) Eliminar la fila anómala (CHIP nulo): todo predio debe tener CHIP.
    out = out.dropna(subset=['PRECHIP']).copy()

    # 2) Códigos catastrales a texto (identificadores, no números).
    for c in COLS_CODIGO:
        out[c] = out[c].astype(str)

    # 3) Imputación lógica de nulos masivos -> 0 ("no aplica").
    for c in COLS_INDUSTRIALES + COLS_ACABADOS:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce').fillna(0)

    # 4) Variables numéricas clave a float.
    for c in ['PREATERRE', 'PREACONST', 'PREPUNTAJE'] + COLS_ESTRUCTURA:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors='coerce')

    return out

DF_LIMPIO = limpiar(DF)
print(f"Filas tras limpieza: {len(DF_LIMPIO):,}  (se eliminó la fila con CHIP nulo si existía)")


### 2.2 Ingeniería de variables (features derivadas)

Se crean dos variables con interpretación de dominio:

- **`IND_CONST`** = `PREACONST / PREATERRE` → intensidad de ocupación / densificación vertical.
- **`CALIDAD_EST`** = media de `PREEARMAZ`, `PREEMUROS`, `PREECUBIER` → calidad estructural consolidada (reduce multicolinealidad).


In [ ]:
def ingenieria(df):
    out = df.copy()
    out['IND_CONST'] = out['PREACONST'] / out['PREATERRE'].replace(0, np.nan)
    out['CALIDAD_EST'] = out[COLS_ESTRUCTURA].mean(axis=1)
    return out

DF_FEAT = ingenieria(DF_LIMPIO)
print("Variables derivadas creadas:", ["IND_CONST", "CALIDAD_EST"])
DF_FEAT[['PREATERRE', 'PREACONST', 'IND_CONST', 'CALIDAD_EST']].describe().T.round(3)


### 2.3 Filtrado de atípicos y preparación de la matriz de modelado


In [ ]:
# Filtrar predios con áreas razonables y puntaje válido (excluye -1 y nulos)
DF_MODEL = DF_FEAT[
    (DF_FEAT['PREATERRE'] > AREA_MIN) & (DF_FEAT['PREATERRE'] < AREA_MAX) &
    (DF_FEAT['PREACONST'] > AREA_MIN) & (DF_FEAT['PREACONST'] < AREA_MAX) &
    (DF_FEAT['PREPUNTAJE'] > PUNTAJE_MIN)
].copy()
DF_MODEL = DF_MODEL.dropna(subset=['PREATERRE', 'PREACONST', 'PREPUNTAJE'] + COLS_ESTRUCTURA)

print(f"Filas aptas para modelado: {len(DF_MODEL):,}  (de {len(DF_FEAT):,})")

# Variable objetivo de clasificación binaria: 1 = residencial, 0 = no residencial
DF_MODEL['ES_RESIDENCIAL'] = (DF_MODEL[TARGET_CLASIFICACION] == CLASE_RESIDENCIAL).astype(int)
print("\nDistribución de la clase (1 = residencial):")
print(DF_MODEL['ES_RESIDENCIAL'].value_counts())


### 2.4 Partición de datos (sin fuga de datos)

- **Regresión:** partición aleatoria 80/20.
- **Clasificación:** partición **estratificada** (`StratifiedShuffleSplit`) para conservar las proporciones de clase ante el desbalance.
- Todo escalamiento/codificación se **ajusta solo sobre el entrenamiento** y luego se aplica a la prueba.


In [ ]:
# --- Muestra opcional para agilizar el entrenamiento en Colab -----------------
def subsample(df, n=MAX_ROWS_FOR_MODELING, stratify=None, seed=RANDOM_STATE):
    if n is None or n >= len(df):
        return df
    if stratify is not None:
        # Muestreo estratificado manual (compatible con cualquier versión de pandas)
        partes = []
        for _, grupo in df.groupby(stratify):
            k = max(1, int(n * len(grupo) / len(df)))
            partes.append(grupo.sample(n=min(k, len(grupo)), random_state=seed))
        return pd.concat(partes)
    return df.sample(n=n, random_state=seed)

DF_REG = subsample(DF_MODEL.copy(), stratify=None)
DF_CLS = subsample(DF_MODEL.copy(), stratify='ES_RESIDENCIAL')

print(f"Muestra regresión : {len(DF_REG):,} filas")
print(f"Muestra clasif.   : {len(DF_CLS):,} filas")


In [ ]:
# --- Regresión: partición 80/20 ---------------------------------------------
X_reg = DF_REG.drop(columns=[TARGET_REGRESION])
y_reg = DF_REG[TARGET_REGRESION].astype(float)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(
    X_reg, y_reg, test_size=TEST_SIZE, random_state=RANDOM_STATE,
)

# --- Clasificación: partición estratificada ---------------------------------
X_cls = DF_CLS.drop(columns=[TARGET_CLASIFICACION, 'ES_RESIDENCIAL'])
y_cls = DF_CLS['ES_RESIDENCIAL']
sss = StratifiedShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
tr_idx, te_idx = next(sss.split(X_cls, y_cls))
Xc_tr, Xc_te, yc_tr, yc_te = X_cls.iloc[tr_idx], X_cls.iloc[te_idx], y_cls.iloc[tr_idx], y_cls.iloc[te_idx]

print("Regresión     : train", Xr_tr.shape, "test", Xr_te.shape)
print("Clasificación : train", Xc_tr.shape, "test", Xc_te.shape)
print("\nProporción de clase (train):", round(yc_tr.mean(), 3), "(test):", round(yc_te.mean(), 3))


### 2.5 Modelo base (baseline)

Como umbral mínimo de rendimiento se entrena una **Regresión Lineal Simple** (solo área de terreno) y una **Regresión Logística** básica para clasificación.


In [ ]:
# Baseline regresión: PREACONST ~ PREATERRE
from sklearn.linear_model import LinearRegression, LogisticRegression

bl = LinearRegression()
bl.fit(Xr_tr[['PREATERRE']], yr_tr)
bl_pred = bl.predict(Xr_te[['PREATERRE']])
print("=== Baseline Regresión Lineal Simple ===")
print(f"R²   : {r2_score(yr_te, bl_pred):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(yr_te, bl_pred)):,.2f}")
print(f"MAE  : {mean_absolute_error(yr_te, bl_pred):,.2f}")
print(f"Pendiente: {bl.coef_[0]:.4f} | Intercepto: {bl.intercept_:.4f}")

# Baseline clasificación: Regresión Logística con variables numéricas
cat_cols = [c for c in ['PRECLASE', 'PREUCALIF'] if c in Xc_tr.columns]
pre = ColumnTransformer([
    ('num', RobustScaler(), FEATURES_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
], remainder='drop')
Xb_tr = pre.fit_transform(Xc_tr, yc_tr)
Xb_te = pre.transform(Xc_te)
blc = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
blc.fit(Xb_tr, yc_tr)
blc_pred = blc.predict(Xb_te)
print("\n=== Baseline Regresión Logística ===")
print(f"Accuracy : {accuracy_score(yc_te, blc_pred):.4f}")
print(f"F1-Score : {f1_score(yc_te, blc_pred):.4f}")
print(f"AUC-ROC  : {roc_auc_score(yc_te, blc.predict_proba(Xb_te)[:, 1]):.4f}")


## FASE 3 — Modelado Avanzado, Clustering y Evaluación

### 3.1 Regresión (predicción del área construida)

Se comparan: regresión lineal multivariable, modelos regularizados (Ridge/Lasso/ElasticNet) y ensambles de árboles (Random Forest, Gradient Boosting). Las variables numéricas se escalan con `RobustScaler` y las categóricas se codifican con One-Hot, todo ajustado **solo sobre entrenamiento**.


In [ ]:
# Preprocesador para regresión
cat_cols_reg = [c for c in ['PRECLASE', 'PREUCALIF'] if c in Xr_tr.columns]
pre_reg = ColumnTransformer([
    ('num', RobustScaler(), FEATURES_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols_reg),
], remainder='drop')

Xr_tr_enc = pre_reg.fit_transform(Xr_tr)
Xr_te_enc = pre_reg.transform(Xr_te)
print("Matriz de entrenamiento (regresión):", Xr_tr_enc.shape)


In [ ]:
modelos_reg = {
    'Regresión Lineal': LinearRegression(),
    'Ridge (L2)': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'Lasso (L1)': Lasso(alpha=0.1, random_state=RANDOM_STATE, max_iter=5000),
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=RANDOM_STATE, max_iter=5000),
    'Random Forest': RandomForestRegressor(n_estimators=150, n_jobs=-1, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=RANDOM_STATE),
}

resultados_reg = []
for nombre, modelo in modelos_reg.items():
    modelo.fit(Xr_tr_enc, yr_tr)
    p = modelo.predict(Xr_te_enc)
    resultados_reg.append({
        'Modelo': nombre,
        'R²': r2_score(yr_te, p),
        'RMSE': np.sqrt(mean_squared_error(yr_te, p)),
        'MAE': mean_absolute_error(yr_te, p),
    })

res_reg = pd.DataFrame(resultados_reg).sort_values('R²', ascending=False).reset_index(drop=True)
res_reg


### 3.2 Clasificación (predicción del destino económico)

Se comparan **varios algoritmos distintos** (regresión logística, KNN, árbol de decisión, Random Forest y Gradient Boosting) con manejo del desbalance mediante `class_weight='balanced'`. Se usa la misma estrategia de preprocesamiento sin fuga de datos.


In [ ]:
cat_cols_cls = [c for c in ['PRECLASE', 'PREUCALIF'] if c in Xc_tr.columns]
pre_cls = ColumnTransformer([
    ('num', RobustScaler(), FEATURES_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols_cls),
], remainder='drop')

Xc_tr_enc = pre_cls.fit_transform(Xc_tr, yc_tr)
Xc_te_enc = pre_cls.transform(Xc_te)
print("Matriz de entrenamiento (clasificación):", Xc_tr_enc.shape)


In [ ]:
modelos_cls = {
    'Regresión Logística': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=12, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=150, class_weight='balanced', n_jobs=-1, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.1, max_depth=4, random_state=RANDOM_STATE),
}

resultados_cls = []
modelos_ajustados_cls = {}
for nombre, modelo in modelos_cls.items():
    modelo.fit(Xc_tr_enc, yc_tr)
    modelos_ajustados_cls[nombre] = modelo
    pred = modelo.predict(Xc_te_enc)
    try:
        auc = roc_auc_score(yc_te, modelo.predict_proba(Xc_te_enc)[:, 1])
    except Exception:
        auc = np.nan
    resultados_cls.append({
        'Modelo': nombre,
        'Accuracy': accuracy_score(yc_te, pred),
        'Precision': precision_score(yc_te, pred, zero_division=0),
        'Recall': recall_score(yc_te, pred, zero_division=0),
        'F1-Score': f1_score(yc_te, pred, zero_division=0),
        'AUC-ROC': auc,
    })

res_cls = pd.DataFrame(resultados_cls).sort_values('F1-Score', ascending=False).reset_index(drop=True)
res_cls


In [ ]:
# Matrices de confusión de los 2 mejores modelos de clasificación
top2 = list(modelos_ajustados_cls.items())[:2]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (nombre, modelo) in zip(axes, top2):
    pred = modelo.predict(Xc_te_enc)
    ConfusionMatrixDisplay.from_predictions(yc_te, pred, ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(nombre, fontsize=9)
plt.tight_layout()
plt.show()


### 3.3 Clustering (segmentación no supervisada)

Se descubren tipologías prediales ocultas. Primero se selecciona el **K óptimo** con el método del **Codo** y el **Coeficiente de Silueta**, y después se entrena un **segundo algoritmo no supervisado** (DBSCAN o jerárquico aglomerativo).


In [ ]:
# Preparación de la matriz de clustering (muestra para agilizar)
DF_CLU = subsample(DF_MODEL.copy(), n=100_000, stratify=None)
X_clu = DF_CLU[FEATURES_CLUSTER].dropna().copy()

scaler_clu = StandardScaler()
X_clu_scaled = scaler_clu.fit_transform(X_clu)
print("Matriz de clustering:", X_clu_scaled.shape)


In [ ]:
# Método del Codo + Coeficiente de Silueta
k_range = range(2, 11)
inercias, siluetas = [], []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_clu_scaled)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_clu_scaled, labels, sample_size=20_000))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(k_range), inercias, marker='o', linestyle='--')
axes[0].set_title('Método del Codo'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inercia (WCSS)')
axes[0].set_xticks(list(k_range)); axes[0].grid(alpha=0.3)
axes[1].plot(list(k_range), siluetas, marker='s', color='g')
axes[1].set_title('Coeficiente de Silueta'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].set_xticks(list(k_range)); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

K_OPTIMO = int(np.nanargmax(siluetas) + k_range.start)
print(f"K óptimo según silueta: {K_OPTIMO}  (silueta = {max(siluetas):.4f})")


In [ ]:
# K-Means final y perfil de los clústeres
kmeans = KMeans(n_clusters=K_OPTIMO, random_state=RANDOM_STATE, n_init=10)
labels_kmeans = kmeans.fit_predict(X_clu_scaled)

perfil = X_clu.copy()
perfil['Cluster'] = labels_kmeans
perfil_grupo = perfil.groupby('Cluster')[FEATURES_CLUSTER].mean().round(2)
perfil_grupo['Cantidad_Predios'] = perfil.groupby('Cluster').size()
display(perfil_grupo)
print(f"Silhouette (K-Means, K={K_OPTIMO}): {silhouette_score(X_clu_scaled, labels_kmeans, sample_size=20_000):.4f}")


In [ ]:
# Segundo algoritmo no supervisado: DBSCAN y Agglomerative (jerárquico)
# DBSCAN (ajusta eps/min_samples según la escala de tus datos)
dbscan = DBSCAN(eps=1.2, min_samples=20)
labels_db = dbscan.fit_predict(X_clu_scaled)
n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
print(f"DBSCAN: {n_clusters_db} clústeres  |  ruido (outliers): {(labels_db == -1).sum():,}")

# Agglomerative (jerárquico) con el mismo K óptimo
agg = AgglomerativeClustering(n_clusters=K_OPTIMO)
labels_agg = agg.fit_predict(X_clu_scaled)
sil_agg = silhouette_score(X_clu_scaled, labels_agg, sample_size=20_000)
print(f"Agglomerative (K={K_OPTIMO}): silueta = {sil_agg:.4f}")


### 3.4 Resumen de resultados y conclusiones


In [ ]:
print("=== MEJORES MODELOS ===")
print("\n[Regresión]")
display(res_reg.head(3))
print("\n[Clasificación]")
display(res_cls.head(3))
print(f"\n[Clustering] K-Means con K={K_OPTIMO} (silueta={silhouette_score(X_clu_scaled, labels_kmeans, sample_size=20_000):.4f})")
print("Interpretación de perfiles:")
print(perfil_grupo.to_string())


## FASE 4 — Exportación para el dashboard y cierre

Los modelos ajustados y los datos agregados pueden guardarse para alimentar el dashboard interactivo (`dashboard/app.py`, construido con Streamlit). El repositorio incluye:

```
proyecto_datos_urbanos/
├── data/               # dataset (completo NO versionado) + muestra
├── notebooks/          # este notebook
├── src/                # código modular reutilizable
├── dashboard/          # app.py de Streamlit
└── requirements.txt    # dependencias exactas
```


In [ ]:
# Guardar artefactos (opcional, para el dashboard)
import pickle, os
os.makedirs('artefactos', exist_ok=True)

mejor_cls_nombre = res_cls.iloc[0]['Modelo']
mejor_cls = modelos_ajustados_cls[mejor_cls_nombre]

with open('artefactos/modelo_clasificacion.pkl', 'wb') as f:
    pickle.dump({'modelo': mejor_cls, 'preprocesador': pre_cls}, f)

perfil_grupo.to_csv('artefactos/perfil_clusters.csv')
print("Artefactos guardados en /artefactos/")
